# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Modelos segmentados por especialidad clínica (APR MDC)

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno evalúa si ajustar modelos independientes por especialidad clínica mejora la predicción de
la DEH. La segmentación se opera mediante la categoría diagnóstica mayor (APR MDC) y se comparan las
tres familias de algoritmos empleadas en el resto del proyecto.

**Los modelos globales de referencia se reutilizan del cuaderno 09**, en lugar de reentrenarse. Esto
ahorra tres ajustes sobre la población completa y, más importante, garantiza que la referencia sea
exactamente la misma en ambos experimentos de segmentación, condición necesaria para que sus resultados
sean comparables entre sí.

## 1. Fundamento

### 1.1 Por qué APR MDC

La categoría diagnóstica mayor agrupa los episodios en aproximadamente dos docenas de clases que
corresponden a sistemas orgánicos y áreas de especialidad: circulatorio, respiratorio, digestivo,
musculoesquelético, nervioso, obstetricia, neonatología y trastornos mentales, entre otras. Es la
agregación disponible en SPARCS más próxima a la noción de especialidad clínica, y opera en un nivel de
granularidad intermedio: suficientemente fino para separar regímenes de estancia muy distintos, y
suficientemente grueso para que cada segmento conserve observaciones bastantes como para ajustar un
modelo.

La expectativa es que la ganancia supere a la de la segmentación etaria del cuaderno 09, dado que el
diagnóstico principal determina la duración de la hospitalización de forma más directa que la edad: un
trasplante cardíaco y una amigdalitis difieren en su estancia esperada mucho más de lo que difieren dos
pacientes de edades distintas con el mismo diagnóstico.

### 1.2 Limitación de la fuente

Las variables `CCSR Diagnosis` y `APR DRG`, que permitirían una segmentación más fina, fueron
transformadas mediante *Target Encoding* en la capa \emph{Gold} y quedaron reducidas a una columna
numérica por variable, de modo que sus categorías originales no son recuperables desde ese estado. Una
segmentación por diagnóstico específico exigiría partir de la capa \emph{Silver}. APR MDC, en cambio,
se codificó mediante indicadores binarios y sí admite reconstrucción.

### 1.3 Nota sobre la validez temporal

La asignación del APR-DRG, y con ella la de la MDC, se realiza durante la codificación del episodio y no
en el momento del ingreso. Esta circunstancia no introduce una dependencia distinta de la ya presente en
el conjunto de predictores —`APR DRG` y `APR Severity of Illness` comparten el mismo origen y se emplean
como variables de entrada en todos los modelos del proyecto—, pero debe declararse al interpretar los
resultados: la partición se apoya en una clasificación administrativa derivada del diagnóstico principal
codificado, no en el diagnóstico presuntivo de admisión.

### 1.4 Advertencia sobre la comparación

El $R^2$ se define respecto de la varianza del desenlace dentro del conjunto evaluado, por lo que el de
una especialidad **no** es comparable con el de otra ni con el del modelo global sobre toda la
población. La comparación válida es pareada dentro de cada segmento —el error del modelo global
restringido a esa especialidad frente al del modelo específico— y, para el juicio de conjunto, la
recomposición de todas las predicciones en un vector único.

## 2. Configuración y carga

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, time, re

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "10"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]


def sanear(nombres):
    """Convierte nombres de columna a caracteres seguros, evitando colisiones."""
    limpios, vistos = [], {}
    for n in nombres:
        s = re.sub(r"[^0-9a-zA-Z_]+", "_", str(n)).strip("_")
        if not s:
            s = "col"
        if s[0].isdigit():
            s = "v_" + s
        if s in vistos:
            vistos[s] += 1
            s = f"{s}__{vistos[s]}"
        else:
            vistos[s] = 0
        limpios.append(s)
    return limpios


NOMBRES_ORIGINALES = list(X_train.columns)
nombres_saneados = sanear(NOMBRES_ORIGINALES)
MAPA_NOMBRES = dict(zip(nombres_saneados, NOMBRES_ORIGINALES))
X_train.columns = nombres_saneados
X_test.columns = nombres_saneados

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Columnas duplicadas tras saneamiento: "
      f"{len(nombres_saneados) - len(set(nombres_saneados))}")


def metricas(y_real, y_pred):
    y_pred = np.clip(y_pred, 1, None)
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
            "R2": r2_score(y_real, y_pred)}

X_train: (1672271, 189)   X_test: (418068, 189)
Columnas duplicadas tras saneamiento: 0


## 3. Reconstrucción de la especialidad (APR MDC)

Al igual que `Age Group`, la variable se codificó con supresión de la primera categoría, de modo que la
categoría de referencia no tiene columna propia y se identifica por la ausencia de todas las demás.

In [2]:
cols_mdc = [col for col in X_train.columns if col.lower().startswith("apr_mdc")]
print(f"Indicadores de APR MDC detectados: {len(cols_mdc)}\n")
for col in cols_mdc[:6]:
    print(f"  {col}   ->   {MAPA_NOMBRES[col]}")
print("  ...")

if not cols_mdc:
    raise ValueError("No se detectaron columnas de APR MDC. Revisar el prefijo.")

ETIQUETA_REF_MDC = "MDC de referencia (categoría suprimida)"


def reconstruir(X, columnas, etiqueta_ref):
    """Recupera una variable categórica a partir de sus indicadores binarios."""
    etiquetas = pd.Series(etiqueta_ref, index=X.index, dtype=object)
    for col in columnas:
        etiquetas[X[col] == 1] = MAPA_NOMBRES[col].split("_", 1)[-1]
    return etiquetas


mdc_train = reconstruir(X_train, cols_mdc, ETIQUETA_REF_MDC)
mdc_test = reconstruir(X_test, cols_mdc, ETIQUETA_REF_MDC)

conteo = pd.DataFrame({
    "n_train": mdc_train.value_counts(),
    "pct_train": mdc_train.value_counts(normalize=True) * 100,
    "n_test": mdc_test.value_counts(),
}).sort_values("n_train", ascending=False)
display(conteo.round(2))
guardar_tabla(conteo, "ConteoPorEspecialidad")

print(f"\nEspecialidades distintas: {mdc_train.nunique()}")
print("Identificar la categoría de referencia contrastando con el conteo de la capa Silver.")

Indicadores de APR MDC detectados: 23

  APR_MDC_11_DISEASES_AND_DISORDERS_OF_THE_KIDNEY_AND_URINARY_TRACT   ->   APR MDC_11: DISEASES AND DISORDERS OF THE KIDNEY AND URINARY TRACT
  APR_MDC_12_DISEASES_AND_DISORDERS_OF_THE_MALE_REPRODUCTIVE_SYSTEM   ->   APR MDC_12: DISEASES AND DISORDERS OF THE MALE REPRODUCTIVE SYSTEM
  APR_MDC_13_DISEASES_AND_DISORDERS_OF_THE_FEMALE_REPRODUCTIVE_SYSTEM   ->   APR MDC_13: DISEASES AND DISORDERS OF THE FEMALE REPRODUCTIVE SYSTEM
  APR_MDC_14_PREGNANCY_CHILDBIRTH_AND_THE_PUERPERIUM   ->   APR MDC_14: PREGNANCY, CHILDBIRTH AND THE PUERPERIUM
  APR_MDC_15_NEWBORNS_AND_OTHER_NEONATES_WITH_CONDITIONS_ORIGINATING_IN_THE_PERINATAL_PERIOD   ->   APR MDC_15: NEWBORNS AND OTHER NEONATES WITH CONDITIONS ORIGINATING IN THE PERINATAL PERIOD
  APR_MDC_16_DISEASES_AND_DISORDERS_OF_THE_BLOOD_AND_BLOOD_FORMING_ORGANS_AND_IMMUNOLOGICAL_DISORDERS   ->   APR MDC_16: DISEASES AND DISORDERS OF THE BLOOD AND BLOOD FORMING ORGANS AND IMMUNOLOGICAL DISORDERS
  ...


,n_train,pct_train,n_test
5: DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,208602,12.47,52656
"14: PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",174421,10.43,43729
4: DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM,170160,10.18,42410
15: NEWBORNS AND OTHER NEONATES WITH CONDITIONS ORIGINATING IN THE PERINATAL PERIOD,159265,9.52,39905
6: DISEASES AND DISORDERS OF THE DIGESTIVE SYSTEM,129274,7.73,32231
8: DISEASES AND DISORDERS OF THE MUSCULOSKELETAL SYSTEM AND CONNECTIVE TISSUE,127358,7.62,31733
18: INFECTIOUS AND PARASITIC DISEASES (SYSTEMIC OR UNSPECIFIED SITES),126551,7.57,31559
1: DISEASES AND DISORDERS OF THE NERVOUS SYSTEM,113128,6.76,28340
11: DISEASES AND DISORDERS OF THE KIDNEY AND URINARY TRACT,73167,4.38,18163
19: MENTAL DISEASES AND DISORDERS,72083,4.31,17856


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_ConteoPorEspecialidad.csv

Especialidades distintas: 24
Identificar la categoría de referencia contrastando con el conteo de la capa Silver.


## 4. Perfil de la estancia por especialidad

Este diagnóstico justifica el experimento antes de entrenar ningún modelo. La dispersión de las medias
entre especialidades —previsiblemente mayor que entre grupos etarios— indica cuánta heterogeneidad puede
capturar la segmentación.

In [3]:
perfil = pd.DataFrame({
    "n": mdc_train.value_counts(),
    "media": y_train.groupby(mdc_train).mean(),
    "mediana": y_train.groupby(mdc_train).median(),
    "desv_std": y_train.groupby(mdc_train).std(),
    "p95": y_train.groupby(mdc_train).quantile(0.95),
    "asimetria": y_train.groupby(mdc_train).skew(),
    "pct_larga": (y_train > 2).groupby(mdc_train).mean() * 100,
}).sort_values("media", ascending=False)

display(perfil.round(2))
guardar_tabla(perfil, "PerfilDEHPorEspecialidad")

cv = perfil["media"].std() / perfil["media"].mean()
print(f"\nRango de medias: {perfil['media'].min():.2f} a {perfil['media'].max():.2f} días")
print(f"Coeficiente de variación de las medias entre especialidades: {cv:.3f}")
print("\nContrastar con la tabla 09_PerfilDEHPorGrupoEtario: una dispersión mayor sugiere")
print("que la partición diagnóstica captura más heterogeneidad que la etaria.")

,n,media,mediana,desv_std,p95,asimetria,pct_larga
19: MENTAL DISEASES AND DISORDERS,72083,12.43,8.0,14.18,36.0,3.69,87.79
"17: MYELOPROLIFERATIVE DISEASES AND DISORDERS, AND POORLY DIFFERENTIATED NEOPLASM",16526,10.37,5.0,13.09,35.0,3.30,82.16
23: FACTORS INFLUENCING HEALTH STATUS AND OTHER CONTACTS WITH HEALTH SERVICES,20736,9.75,7.0,10.15,26.0,3.58,76.09
25: MULTIPLE SIGNIFICANT TRAUMA,4390,9.32,6.0,10.08,27.0,3.70,83.74
18: INFECTIOUS AND PARASITIC DISEASES (SYSTEMIC OR UNSPECIFIED SITES),126551,9.29,6.0,10.84,28.0,4.06,84.15
22: BURNS,1393,8.64,5.0,12.17,30.0,3.73,67.19
4: DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM,170160,6.85,4.0,9.10,21.0,5.28,72.24
1: DISEASES AND DISORDERS OF THE NERVOUS SYSTEM,113128,6.78,4.0,9.43,22.0,5.02,65.71
20: ALCOHOL/DRUG USE AND ALCOHOL/DRUG INDUCED ORGANIC MENTAL DISORDERS,46212,6.10,4.0,7.02,25.0,2.94,69.68
7: DISEASES AND DISORDERS OF THE HEPATOBILIARY SYSTEM AND PANCREAS,49102,5.53,4.0,6.87,16.0,6.02,68.34


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_PerfilDEHPorEspecialidad.csv

Rango de medias: 2.69 a 12.43 días
Coeficiente de variación de las medias entre especialidades: 0.425

Contrastar con la tabla 09_PerfilDEHPorGrupoEtario: una dispersión mayor sugiere
que la partición diagnóstica captura más heterogeneidad que la etaria.


## 5. Referencia global: reutilización de los modelos del cuaderno 09

En lugar de reentrenar las tres familias sobre la población completa, se cargan los modelos ya ajustados
en el cuaderno 09 y se obtienen sus predicciones sobre el conjunto de prueba. Además del ahorro
computacional, esto garantiza que la referencia sea idéntica en ambos experimentos de segmentación y que
sus resultados resulten directamente comparables.

Si los archivos no existen —por no haberse ejecutado aún el cuaderno 09—, la celda entrena los modelos
globales y los persiste, dejando constancia de ello en la salida.

In [4]:
FRACCION = 0.30   # debe coincidir con la empleada en el cuaderno 09
SEMILLA = 42

if FRACCION < 1.0:
    idx = X_train.sample(frac=FRACCION, random_state=SEMILLA).index
    X_tr, y_tr, mdc_tr = X_train.loc[idx], y_train.loc[idx], mdc_train.loc[idx]
else:
    X_tr, y_tr, mdc_tr = X_train, y_train, mdc_train

print(f"Entrenamiento: {X_tr.shape[0]:,} filas   |   Prueba: {X_test.shape[0]:,} filas\n")

FAMILIAS = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=20, random_state=SEMILLA, n_jobs=-1),
    "XGBoost Tweedie": XGBRegressor(
        objective="reg:tweedie", tweedie_variance_power=1.5,
        n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, tree_method="hist", random_state=SEMILLA, n_jobs=-1),
}

pred_global, filas_glob = {}, []

for fam, plantilla in FAMILIAS.items():
    slug = re.sub(r"[^0-9a-zA-Z]+", "_", fam)
    ruta = MODELOS_PATH / f"09_global_{slug}.joblib"

    if ruta.exists():
        modelo = joblib.load(ruta)
        origen = "reutilizado del cuaderno 09"
    else:
        print(f"  [{fam}] No se encontró {ruta.name}; se entrena el modelo global.")
        modelo = clone(plantilla)
        modelo.fit(X_tr, y_tr)
        joblib.dump(modelo, MODELOS_PATH / f"10_global_{slug}.joblib")
        origen = "entrenado en este cuaderno"

    pred_global[fam] = np.clip(modelo.predict(X_test), 1, None)
    filas_glob.append({"familia": fam, "origen": origen,
                       **metricas(y_test, pred_global[fam])})
    print(f"{fam:18s}  MAE={filas_glob[-1]['MAE']:.3f}  R2={filas_glob[-1]['R2']:.4f}  "
          f"({origen})")

glob_df = pd.DataFrame(filas_glob)
display(glob_df.round(4))
guardar_tabla(glob_df, "ReferenciaGlobal")

Entrenamiento: 501,681 filas   |   Prueba: 418,068 filas

Ridge               MAE=3.267  R2=0.3837  (reutilizado del cuaderno 09)
Random Forest       MAE=2.994  R2=0.4263  (reutilizado del cuaderno 09)
XGBoost Tweedie     MAE=2.910  R2=0.4535  (reutilizado del cuaderno 09)


,familia,origen,MAE,RMSE,R2
0,Ridge,reutilizado del cuaderno 09,3.2673,6.5174,0.3837
1,Random Forest,reutilizado del cuaderno 09,2.9942,6.2883,0.4263
2,XGBoost Tweedie,reutilizado del cuaderno 09,2.9103,6.1371,0.4535


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_ReferenciaGlobal.csv


## 6. Modelos por especialidad

El número de especialidades hace que el costo crezca rápidamente. Se aplican dos controles: solo se
ajusta modelo propio a las categorías con al menos `MIN_OBS` observaciones de entrenamiento, y `TOP_N`
limita el experimento a las de mayor volumen. Las restantes se respaldan con la predicción del modelo
global, de modo que el vector de predicciones segmentadas quede completo sobre el conjunto de prueba.

Una consideración sobre la estructura del problema: dentro de un segmento definido por MDC, los
indicadores binarios de esa misma variable se vuelven constantes y dejan de aportar información, y el
rango de `APR DRG_TE` se estrecha considerablemente. El modelo específico dispone, por tanto, de un
espacio de variación menor que el global, lo que constituye parte del costo de la especialización.

In [5]:
MIN_OBS = 2000
TOP_N = 12   # None para incluir todas las especialidades que superen MIN_OBS

orden = mdc_tr.value_counts()
candidatas = [g for g in orden.index if orden[g] >= MIN_OBS]
if TOP_N is not None:
    candidatas = candidatas[:TOP_N]

print(f"Especialidades con modelo propio: {len(candidatas)} de {mdc_tr.nunique()}")
print(f"Cobertura del entrenamiento: {orden[candidatas].sum() / len(mdc_tr) * 100:.1f}%\n")

pred_segmentada = {}
filas_seg = []

for fam, plantilla in FAMILIAS.items():
    print(f"=== Familia: {fam} ===")
    # Respaldo global en todo el conjunto; se sobrescribe donde haya modelo propio
    pred_segmentada[fam] = pd.Series(pred_global[fam], index=X_test.index, dtype=float)

    for g in candidatas:
        m_tr = (mdc_tr == g).to_numpy()
        m_te = (mdc_test == g).to_numpy()
        if m_te.sum() == 0:
            continue

        t0 = time.time()
        modelo = clone(plantilla)
        modelo.fit(X_tr[m_tr], y_tr[m_tr])
        p_seg = np.clip(modelo.predict(X_test[m_te]), 1, None)
        pred_segmentada[fam][m_te] = p_seg

        y_seg = y_test[m_te]
        m_glob = metricas(y_seg, pred_global[fam][m_te])
        m_seg = metricas(y_seg, p_seg)

        filas_seg.append({
            "familia": fam, "especialidad": g[:45],
            "n_train": int(m_tr.sum()), "n_test": int(m_te.sum()),
            "dias_medios": float(y_seg.mean()),
            "MAE_global": m_glob["MAE"], "MAE_segmentado": m_seg["MAE"],
            "mejora_MAE_pct": (m_glob["MAE"] - m_seg["MAE"]) / m_glob["MAE"] * 100,
            "R2_global": m_glob["R2"], "R2_segmentado": m_seg["R2"],
        })
        print(f"  {g[:42]:44s} MAE {m_glob['MAE']:.3f} -> {m_seg['MAE']:.3f}  "
              f"({filas_seg[-1]['mejora_MAE_pct']:+.1f}%)  ({time.time()-t0:.0f}s)")
    print()

comp_seg = pd.DataFrame(filas_seg)
display(comp_seg.round(3))
guardar_tabla(comp_seg, "ComparacionPorEspecialidad")

Especialidades con modelo propio: 12 de 24
Cobertura del entrenamiento: 87.5%

=== Familia: Ridge ===
  5: DISEASES AND DISORDERS OF THE CIRCULATO   MAE 3.000 -> 2.910  (+3.0%)  (0s)
  14: PREGNANCY, CHILDBIRTH AND THE PUERPERI   MAE 1.150 -> 0.957  (+16.8%)  (1s)
  4: DISEASES AND DISORDERS OF THE RESPIRATO   MAE 4.206 -> 4.024  (+4.3%)  (0s)
  15: NEWBORNS AND OTHER NEONATES WITH CONDI   MAE 1.623 -> 1.513  (+6.8%)  (0s)
  6: DISEASES AND DISORDERS OF THE DIGESTIVE   MAE 2.889 -> 2.789  (+3.4%)  (0s)
  8: DISEASES AND DISORDERS OF THE MUSCULOSK   MAE 2.743 -> 2.649  (+3.4%)  (0s)
  18: INFECTIOUS AND PARASITIC DISEASES (SYS   MAE 5.198 -> 5.168  (+0.6%)  (0s)
  1: DISEASES AND DISORDERS OF THE NERVOUS S   MAE 4.182 -> 4.102  (+1.9%)  (0s)
  19: MENTAL DISEASES AND DISORDERS            MAE 7.775 -> 7.641  (+1.7%)  (0s)
  11: DISEASES AND DISORDERS OF THE KIDNEY A   MAE 3.000 -> 2.958  (+1.4%)  (0s)
  MDC de referencia (categoría suprimida)      MAE 2.301 -> 2.239  (+2.7%)  (0s)
  7: D

,familia,especialidad,n_train,n_test,dias_medios,MAE_global,MAE_segmentado,mejora_MAE_pct,R2_global,R2_segmentado
0,Ridge,5: DISEASES AND DISORDERS OF THE CIRCULATORY,62234,52656,5.234,3.000,2.910,3.015,0.361,0.373
1,Ridge,"14: PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",52364,43729,2.704,1.150,0.957,16.762,0.116,0.195
2,Ridge,4: DISEASES AND DISORDERS OF THE RESPIRATORY,51331,42410,6.860,4.206,4.024,4.334,0.346,0.362
3,Ridge,15: NEWBORNS AND OTHER NEONATES WITH CONDITIO,47936,39905,3.542,1.623,1.513,6.757,0.693,0.723
4,Ridge,6: DISEASES AND DISORDERS OF THE DIGESTIVE SY,38542,32231,5.103,2.889,2.789,3.445,0.312,0.345
5,Ridge,8: DISEASES AND DISORDERS OF THE MUSCULOSKELE,38337,31733,4.978,2.743,2.649,3.421,0.318,0.354
6,Ridge,18: INFECTIOUS AND PARASITIC DISEASES (SYSTEM,37679,31559,9.316,5.198,5.168,0.564,0.331,0.340
7,Ridge,1: DISEASES AND DISORDERS OF THE NERVOUS SYST,34067,28340,6.698,4.182,4.102,1.923,0.308,0.338
8,Ridge,19: MENTAL DISEASES AND DISORDERS,21784,17856,12.524,7.775,7.641,1.713,0.126,0.157
9,Ridge,11: DISEASES AND DISORDERS OF THE KIDNEY AND,21604,18163,5.437,3.000,2.958,1.402,0.257,0.270


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_ComparacionPorEspecialidad.csv


In [6]:
pivote = comp_seg.pivot(index="especialidad", columns="familia", values="mejora_MAE_pct")
print("Mejora porcentual del MAE por especialidad (positivo = la segmentación aporta):")
display(pivote.round(2))
guardar_tabla(pivote, "PivoteMejoraMAE_Especialidad")

print("\nEspecialidades donde las tres familias coinciden en mejorar:")
coinciden = pivote[(pivote > 0).all(axis=1)].index.tolist()
for e in coinciden:
    print(f"  - {e}")
if not coinciden:
    print("  (ninguna)")

Mejora porcentual del MAE por especialidad (positivo = la segmentación aporta):


familia,Random Forest,Ridge,XGBoost Tweedie
especialidad,,,
11: DISEASES AND DISORDERS OF THE KIDNEY AND,-2.51,1.40,-0.03
"14: PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",-1.39,16.76,2.62
15: NEWBORNS AND OTHER NEONATES WITH CONDITIO,-0.30,6.76,1.19
18: INFECTIOUS AND PARASITIC DISEASES (SYSTEM,-2.21,0.56,0.22
19: MENTAL DISEASES AND DISORDERS,-2.59,1.71,0.59
1: DISEASES AND DISORDERS OF THE NERVOUS SYST,-1.71,1.92,1.72
4: DISEASES AND DISORDERS OF THE RESPIRATORY,-0.67,4.33,1.51
5: DISEASES AND DISORDERS OF THE CIRCULATORY,-0.77,3.01,1.83
6: DISEASES AND DISORDERS OF THE DIGESTIVE SY,-1.82,3.44,1.83


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_PivoteMejoraMAE_Especialidad.csv

Especialidades donde las tres familias coinciden en mejorar:
  (ninguna)


Un patrón consistente entre familias —todas mejoran en las mismas especialidades y empeoran en las
mismas— indica que el efecto proviene de la estructura clínica de la población y no de una interacción
particular con algún algoritmo. Si por el contrario solo una familia se beneficia, lo que la
segmentación corrige es una limitación de ese modelo concreto.

## 7. Juicio de conjunto

Las predicciones de todas las especialidades se recomponen en un vector único sobre el conjunto de
prueba completo, lo que permite comparar cada configuración segmentada con su propio modelo global y con
los resultados de los cuadernos anteriores.

In [7]:
filas_final = []
for fam in FAMILIAS:
    assert pred_segmentada[fam].notna().all(), f"Predicciones incompletas en {fam}"
    m_g = metricas(y_test, pred_global[fam])
    m_s = metricas(y_test, pred_segmentada[fam].to_numpy())
    filas_final.append({
        "familia": fam,
        "MAE_global": m_g["MAE"], "MAE_segmentado": m_s["MAE"],
        "delta_MAE": m_g["MAE"] - m_s["MAE"],
        "R2_global": m_g["R2"], "R2_segmentado": m_s["R2"],
        "delta_R2": m_s["R2"] - m_g["R2"],
    })

final_df = pd.DataFrame(filas_final)
display(final_df.round(4))
guardar_tabla(final_df, "ResumenGlobalVsEspecialidad")

mejor = final_df.sort_values("MAE_segmentado").iloc[0]
print(f"\nMejor configuración: {mejor['familia']}  "
      f"(MAE={mejor['MAE_segmentado']:.3f}, R2={mejor['R2_segmentado']:.4f})")

if (final_df["delta_R2"] < 0.005).all() and (final_df["delta_MAE"] < 0.05).all():
    print("\n>>> La ganancia es marginal en las tres familias. La segmentación por especialidad")
    print(">>> no justifica mantener múltiples modelos ni renunciar a una interpretación SHAP")
    print(">>> unificada.")

print("\nContrastar con 09_ResumenGlobalVsSegmentado para establecer cuál de las dos")
print("particiones —diagnóstica o etaria— resulta más informativa.")

,familia,MAE_global,MAE_segmentado,delta_MAE,R2_global,R2_segmentado,delta_R2
0,Ridge,3.2673,3.1709,0.0964,0.3837,0.4024,0.0187
1,Random Forest,2.9942,3.0387,-0.0445,0.4263,0.4058,-0.0205
2,XGBoost Tweedie,2.9103,2.8786,0.0317,0.4535,0.4405,-0.0130


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\10_ResumenGlobalVsEspecialidad.csv

Mejor configuración: XGBoost Tweedie  (MAE=2.879, R2=0.4405)

Contrastar con 09_ResumenGlobalVsSegmentado para establecer cuál de las dos
particiones —diagnóstica o etaria— resulta más informativa.


## 8. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4:

1. **El perfil por especialidad** (Sección 4). El rango de medias y su coeficiente de variación,
   contrastados con los de la partición etaria, establecen cuál de las dos captura más heterogeneidad.
   Esta comparación se realiza entre tablas, sin necesidad de reentrenar.
2. **La comparación pareada** (Sección 6). Es previsible que las especialidades con estancias largas y
   variables —trasplantes, oncología hematológica, neonatología— se beneficien más que las de
   procedimientos estandarizados.
3. **La consistencia entre familias** (Sección 6). Las especialidades en las que las tres familias
   coinciden en mejorar son las que sostienen una conclusión robusta.
4. **El juicio de conjunto** (Sección 7). Determina si la segmentación mejora la predicción sobre la
   población completa y permite contrastar directamente ambas particiones.
5. **La limitación de la fuente** declarada en la Sección 1.2: la segmentación por diagnóstico específico
   requeriría reconstruir el conjunto desde la capa \emph{Silver}, dado que `CCSR Diagnosis` y `APR DRG`
   perdieron sus categorías originales en el *Target Encoding*.
6. **La nota de validez temporal** de la Sección 1.3, sobre el momento en que se asigna la clasificación
   APR.

**Tablas exportadas:** `10_ConteoPorEspecialidad`, `10_PerfilDEHPorEspecialidad`,
`10_ReferenciaGlobal`, `10_ComparacionPorEspecialidad`, `10_PivoteMejoraMAE_Especialidad`,
`10_ResumenGlobalVsEspecialidad`.